# TP2 - Etapa 2: dataset, preprocesamiento, entrenamiento y comparacion de modelos

Notebook de la Etapa 2: analisis del dataset, preprocesamiento, fine-tuning de ResNet18
(Modelo A), CNN propia (Modelo B) y estudio comparativo.


Federico Barbarroja

## 1. Clonamos el repositorio

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO_URL = f"https://{token}@github.com/FedeBarbarroja/tuia-dog-recognition-app.git"

!git clone $REPO_URL proyecto
%cd proyecto

## 2. Instalar dependencias

In [ ]:
!pip install -q pydantic-settings python-dotenv kagglehub albumentations scikit-learn

## 3. Configuracion del entorno

In [ ]:
import os
import sys
from pathlib import Path

os.environ["USE_PGVECTOR"] = "false"

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from lib.config import settings
import torch

print("dataset_path:", settings.dataset_path)
print("GPU disponible:", torch.cuda.is_available())
print("Dispositivo:", "cuda" if torch.cuda.is_available() else "cpu")

## 4. Descargar el dataset

In [ ]:
!python scripts/download_dataset.py
!ls data/dataset

## 5. Analisis del dataset

El dataset **70 Dog Breeds Image Dataset** contiene imagenes de 70 razas organizadas en
tres splits: `train/`, `valid/` y `test/`. Analizamos distribucion de clases, balance
entre splits y ejemplos visuales.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import cv2

DATASET_PATH = Path("data/dataset")
SPLITS = ["train", "valid", "test"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

records = []
for split in SPLITS:
    for breed_dir in sorted((DATASET_PATH / split).iterdir()):
        if not breed_dir.is_dir():
            continue
        count = sum(1 for f in breed_dir.iterdir() if f.suffix.lower() in IMAGE_EXTS)
        records.append({"split": split, "breed": breed_dir.name, "count": count})

df = pd.DataFrame(records)
pivot = df.pivot(index="breed", columns="split", values="count").fillna(0).astype(int)
pivot["total"] = pivot.sum(axis=1)
pivot = pivot.sort_values("total", ascending=False)
breeds = pivot.index.tolist()

print(f"Razas: {len(pivot)}")
print(f"Imagenes totales: {pivot['total'].sum()}")
for split in SPLITS:
    print(f"  {split}: {pivot[split].sum()} imagenes")
print(f"\nMin imagenes por raza (train): {pivot['train'].min()}")
print(f"Max imagenes por raza (train): {pivot['train'].max()}")
print(f"Promedio por raza (train): {pivot['train'].mean():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))
x = np.arange(len(breeds))
ax.bar(x, pivot["train"], color="steelblue", label="train")
ax.bar(x, pivot["valid"], bottom=pivot["train"], color="orange", label="valid")
ax.bar(x, pivot["test"], bottom=pivot["train"] + pivot["valid"], color="green", label="test")
ax.set_xticks(x)
ax.set_xticklabels(breeds, rotation=90, fontsize=7)
ax.set_xlabel("Raza")
ax.set_ylabel("Imagenes")
ax.set_title("Distribucion de imagenes por raza y split")
ax.legend()
plt.tight_layout()
plt.show()

print("\n10 razas con mas imagenes:")
print(pivot[["train", "valid", "test", "total"]].head(10).to_string())
print("\n10 razas con menos imagenes:")
print(pivot[["train", "valid", "test", "total"]].tail(10).to_string())

In [ ]:
sample_breeds = random.sample(breeds, 9)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for ax, breed in zip(axes.flat, sample_breeds):
    img_path = next(f for f in (DATASET_PATH / "train" / breed).iterdir() if f.suffix.lower() in IMAGE_EXTS)
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(breed, fontsize=8)
    ax.axis("off")

plt.suptitle("Ejemplos de razas del dataset")
plt.tight_layout()
plt.show()

### Filtrado de imagenes de baja calidad

Se filtran imagenes que no aportan informacion util para el entrenamiento. El criterio usa
dos condiciones:
- **Resolucion minima**: imagenes menores a 50x50 px son demasiado pequenas para que
  el modelo extraiga features significativas despues del resize a 224x224.
- **Varianza minima**: imagenes con varianza de pixeles menor a 100 son practicamente
  monocromaticas (fondo liso, imagen corrompida o sobreexpuesta) y no aportan
  informacion visual de la raza.

Las imagenes filtradas se listan pero **no se eliminan** del disco — solo se excluyen
del analisis para tener visibilidad del problema sin modificar el dataset original.

In [ ]:
MIN_SIZE = 50       # pixeles minimos en cada dimension
MIN_VARIANCE = 100  # varianza minima de pixeles (escala 0-255)

low_quality = []

for split in SPLITS:
    for breed_dir in (DATASET_PATH / split).iterdir():
        if not breed_dir.is_dir():
            continue
        for img_path in breed_dir.iterdir():
            if img_path.suffix.lower() not in IMAGE_EXTS:
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                low_quality.append({"path": str(img_path), "motivo": "corrupta"})
                continue
            h, w = img.shape[:2]
            if h < MIN_SIZE or w < MIN_SIZE:
                low_quality.append({"path": str(img_path), "motivo": f"resolucion {w}x{h}"})
                continue
            if img.var() < MIN_VARIANCE:
                low_quality.append({"path": str(img_path), "motivo": f"varianza {img.var():.1f}"})

print(f"Imagenes de baja calidad detectadas: {len(low_quality)} de {pivot['total'].sum()} ({len(low_quality)/pivot['total'].sum():.1%})")
if low_quality:
    df_lq = pd.DataFrame(low_quality)
    print("\nDistribucion por motivo:")
    print(df_lq["motivo"].apply(lambda x: x.split()[0]).value_counts().to_string())
    print("\nEjemplos:")
    print(df_lq.head(10).to_string(index=False))
else:
    print("No se detectaron imagenes de baja calidad con los criterios definidos.")

## 6. Preprocesamiento y Data Augmentation

### Resize y normalizacion
Las imagenes se redimensionan a **224x224 px** (tamano estandar de ResNet18 en ImageNet).
La normalizacion usa `mean=[0.485, 0.456, 0.406]` y `std=[0.229, 0.224, 0.225]` — los
valores del dataset ImageNet. ResNet18 fue entrenado con esos valores y sus pesos esperan
esa distribucion de entrada.

### Data Augmentation (solo train)
- **RandomHorizontalFlip**: los perros aparecen en cualquier orientacion horizontal.
- **RandomRotation(15)**: simula distintos angulos de camara sin distorsionar el sujeto.
- **ColorJitter**: simula variaciones de iluminacion y calidad fotografica.
- **RandomGrayscale(p=0.05)**: fuerza al modelo a no depender solo del color para distinguir razas.

En validacion y test no se aplica augmentation para que las metricas sean reproducibles.

In [ ]:
from torchvision import transforms
from PIL import Image

augment_only = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
])

sample_breed = random.choice(breeds)
img_path = next(f for f in (DATASET_PATH / "train" / sample_breed).iterdir() if f.suffix.lower() in IMAGE_EXTS)
pil_img = Image.open(img_path).convert("RGB")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes.flat[0].imshow(pil_img.resize((224, 224)))
axes.flat[0].set_title("Original")
axes.flat[0].axis("off")

for ax in axes.flat[1:]:
    ax.imshow(augment_only(pil_img))
    ax.set_title("Augmented")
    ax.axis("off")

plt.suptitle(f"Data augmentation — {sample_breed}")
plt.tight_layout()
plt.show()

## 7. Modelo A: fine-tuning de ResNet18

Se reemplaza la capa `fc` de ResNet18 (512->1000) por una nueva (512->70 razas).
Se entrena toda la red con Adam y StepLR.

**Hiperparametros:** epochs=20, batch=32, lr=0.001, step=7, gamma=0.1, optimizer=Adam, loss=CrossEntropyLoss

In [ ]:
from lib.bootstrap import build_classifier

classifier = build_classifier(settings)
classifier.set_active_model("resnet18_finetuned")
classifier.train_classifier()

In [ ]:
metrics_a = classifier.evaluate_classifier()
print("Metricas Modelo A (ResNet18 fine-tuned):")
for k, v in metrics_a.items():
    print(f"  {k}: {v:.4f}")

### Curvas de entrenamiento y matriz de confusion (Modelo A)

In [ ]:
history_a = classifier.history
epochs_range = range(1, len(history_a["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_range, history_a["train_loss"], label="Train")
ax1.plot(epochs_range, history_a["val_loss"], label="Val")
ax1.set_title("Loss — ResNet18 fine-tuned")
ax1.set_xlabel("Epoca")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs_range, history_a["train_acc"], label="Train")
ax2.plot(epochs_range, history_a["val_acc"], label="Val")
ax2.set_title("Accuracy — ResNet18 fine-tuned")
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torchvision.datasets import ImageFolder

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_a = classifier.load_model("resnet18_finetuned")
model_a = checkpoint_a["model"].eval().to(device)
classes_a = checkpoint_a["classes"]

test_loader = torch.utils.data.DataLoader(
    ImageFolder("data/dataset/test", transform=val_transform),
    batch_size=32, shuffle=False
)

all_preds_a, all_labels_a = [], []
with torch.no_grad():
    for images, labels in test_loader:
        all_preds_a.extend(model_a(images.to(device)).argmax(1).cpu().tolist())
        all_labels_a.extend(labels.tolist())

cm_a = confusion_matrix(all_labels_a, all_preds_a)
fig, ax = plt.subplots(figsize=(20, 18))
ConfusionMatrixDisplay(cm_a, display_labels=classes_a).plot(ax=ax, colorbar=False, xticks_rotation=90)
ax.set_title("Matriz de confusion — ResNet18 fine-tuned")
plt.tight_layout()
plt.show()

per_class_acc_a = cm_a.diagonal() / cm_a.sum(axis=1)
worst_a = np.argsort(per_class_acc_a)[:5]
print("\nClases con peor accuracy (Modelo A):")
for i in worst_a:
    print(f"  {classes_a[i]}: {per_class_acc_a[i]:.2%}")

## 8. Modelo B: CNN propia

Arquitectura _CustomCNN: 4 bloques Conv+BN+ReLU+MaxPool, AdaptiveAvgPool,
FC 256->512 (embedding layer), FC 512->70 clases. Entrenada desde cero.

In [ ]:
classifier.set_active_model("cnn_custom")
classifier.train_classifier()

In [ ]:
metrics_b = classifier.evaluate_classifier()
print("Metricas Modelo B (CNN custom):")
for k, v in metrics_b.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
history_b = classifier.history
epochs_range_b = range(1, len(history_b["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(epochs_range_b, history_b["train_loss"], label="Train")
ax1.plot(epochs_range_b, history_b["val_loss"], label="Val")
ax1.set_title("Loss — CNN custom")
ax1.set_xlabel("Epoca")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(epochs_range_b, history_b["train_acc"], label="Train")
ax2.plot(epochs_range_b, history_b["val_acc"], label="Val")
ax2.set_title("Accuracy — CNN custom")
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

## 9. Estudio comparativo

In [ ]:
comparison = pd.DataFrame({
    "ResNet18 fine-tuned": metrics_a,
    "CNN custom": metrics_b,
}).T

print("Comparacion de metricas (test set):")
print(comparison.to_string(float_format="{:.4f}".format))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison.columns))
w = 0.35
ax.bar(x - w/2, comparison.loc["ResNet18 fine-tuned"], w, label="ResNet18 fine-tuned")
ax.bar(x + w/2, comparison.loc["CNN custom"], w, label="CNN custom")
ax.set_xticks(x)
ax.set_xticklabels(comparison.columns)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Comparacion de metricas — Modelo A vs Modelo B")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(history_a["val_acc"]) + 1), history_a["val_acc"], label="ResNet18")
ax1.plot(range(1, len(history_b["val_acc"]) + 1), history_b["val_acc"], label="CNN custom")
ax1.set_title("Val accuracy comparada")
ax1.set_xlabel("Epoca")
ax1.set_ylabel("Accuracy")
ax1.legend()

ax2.plot(range(1, len(history_a["val_loss"]) + 1), history_a["val_loss"], label="ResNet18")
ax2.plot(range(1, len(history_b["val_loss"]) + 1), history_b["val_loss"], label="CNN custom")
ax2.set_title("Val loss comparada")
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Loss")
ax2.legend()

plt.tight_layout()
plt.show()

### Analisis de errores y conclusiones

Las razas con peor desempeno suelen ser las visualmente similares (distintos tipos de Terrier,
Retriever, Collie). El modelo confunde estas razas porque sus features visuales son muy parecidas.

**Trade-offs:**
- **ResNet18 fine-tuned**: mejor accuracy, converge rapido (~20 epocas), requiere pesos pre-entrenados (~45MB).
- **CNN custom**: menor accuracy, necesita mas epocas para converger, arquitectura mas liviana y completamente controlable.

**Conclusion**: el transfer learning (Modelo A) es claramente superior para este problema
porque 70 razas con un dataset moderado favorece aprovechar features ya aprendidas en ImageNet.
La CNN propia necesitaria muchos mas datos o una arquitectura mas profunda para competir.

## 10. Evaluacion con conjunto independiente

Se construye un conjunto independiente con imagenes descargadas de internet (fuera del
dataset de Kaggle) para evaluar la capacidad de generalizacion del modelo.

**Como subir las imagenes a Colab:**
Ejecutar la celda siguiente para subir las imagenes manualmente.

Las imagenes deben estar en carpetas por raza:
```
eval_independiente/
  golden_retriever/  ← nombre exacto de la clase en el dataset
    img1.jpg
    img2.jpg
  labrador/
    img1.jpg
```
Si no se conoce la raza exacta, se pueden poner todas en una sola carpeta y evaluar visualmente.

In [ ]:
# Subir imagenes desde tu computadora a Colab
try:
    from google.colab import files
    uploaded = files.upload()  # seleccionar las imagenes de internet

    EVAL_PATH = Path("data/eval_independiente")
    EVAL_PATH.mkdir(parents=True, exist_ok=True)

    for filename, content in uploaded.items():
        (EVAL_PATH / filename).write_bytes(content)

    print(f"Imagenes subidas: {list(EVAL_PATH.iterdir())}")
except ImportError:
    print("Fuera de Colab — colocar imagenes manualmente en data/eval_independiente/")

In [ ]:
# Clasificar las imagenes del conjunto independiente con Modelo A (ResNet18 fine-tuned)
EVAL_PATH = Path("data/eval_independiente")
img_files = [f for f in EVAL_PATH.rglob("*") if f.suffix.lower() in IMAGE_EXTS]

if not img_files:
    print("No se encontraron imagenes en data/eval_independiente/")
else:
    results = []
    model_a.eval()

    for img_path in img_files:
        pil_img = Image.open(img_path).convert("RGB")
        tensor = val_transform(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model_a(tensor)
            probs = torch.softmax(output, dim=1)
            top5 = probs.topk(5)

        pred_breed = classes_a[top5.indices[0][0].item()]
        pred_score = top5.values[0][0].item()
        true_breed = img_path.parent.name if img_path.parent != EVAL_PATH else "desconocida"

        results.append({
            "imagen": img_path.name,
            "raza_real": true_breed,
            "prediccion": pred_breed,
            "confianza": f"{pred_score:.2%}",
            "correcto": pred_breed == true_breed,
        })

    df_eval = pd.DataFrame(results)
    print(df_eval.to_string(index=False))
    if "desconocida" not in df_eval["raza_real"].values:
        acc_indep = df_eval["correcto"].mean()
        print(f"\nAccuracy conjunto independiente: {acc_indep:.2%}")

In [ ]:
# Visualizar predicciones sobre las imagenes independientes
if img_files:
    n = min(len(img_files), 6)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for ax, img_path, result in zip(axes, img_files[:n], results[:n]):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        color = "green" if result["correcto"] else "red"
        ax.set_title(
            f"Pred: {result['prediccion']}\n({result['confianza']})",
            fontsize=8, color=color
        )
        ax.axis("off")

    plt.suptitle("Predicciones sobre conjunto independiente (verde=correcto, rojo=incorrecto)")
    plt.tight_layout()
    plt.show()

## 11. Descargar los checkpoints

In [ ]:
!ls -lh models/

try:
    from google.colab import files
    files.download(str(settings.model_path / settings.resnet18_model_name))
    files.download(str(settings.model_path / settings.cnn_custom_model_name))
except ImportError:
    print("Fuera de Colab: los checkpoints quedan en models/")